# Anforderungen installieren

In [19]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


# Setup & Imports

In [20]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime, date, timedelta
import requests
import math
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib.parse import quote
from io import StringIO
import time
import sys
from psycopg2 import sql
from dotenv import load_dotenv




# Verzeichnisse
DATA_DIR = "./data"
REPORT_DIR = "./reports"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

today = datetime.now().strftime("%Y-%m-%d")
CSV_PATH = os.path.join(DATA_DIR, f"egid_buildings_{today}.csv")
REPORT_PATH = os.path.join(REPORT_DIR, f"updatereport_{today}.txt")


# PostGre DB erstellen inkl. Tabellen oder nur laden falls bereits vorhanden

In [22]:
# === .env-Datei laden ===
load_dotenv()

DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")

# === Verbindung prüfen ===
print(f"🔗 Verbinde mit PostgreSQL → {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cur = conn.cursor()
    print(f"✅ Verbindung zu bestehender Datenbank '{DB_NAME}' hergestellt.")

except psycopg2.OperationalError:
    print(f"⚠️ Datenbank '{DB_NAME}' existiert noch nicht – wird erstellt ...")
    conn = psycopg2.connect(
        dbname="postgres",
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(DB_NAME)))
    print(f"✅ Datenbank '{DB_NAME}' wurde erstellt.")

    # Verbindung zur neuen DB aufbauen
    conn.close()
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cur = conn.cursor()
    print(f"🔁 Verbindung zu neuer Datenbank '{DB_NAME}' hergestellt.")


# === 1️⃣ .env-Datei laden ===
load_dotenv()

DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")

print(f"🔗 Verbinde mit PostgreSQL → {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# === 2️⃣ Verbindung prüfen / DB erstellen falls nicht vorhanden ===
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cur = conn.cursor()
    print(f"✅ Verbindung zu bestehender Datenbank '{DB_NAME}' hergestellt.")

except psycopg2.OperationalError:
    print(f"⚠️ Datenbank '{DB_NAME}' existiert noch nicht – wird erstellt ...")
    conn = psycopg2.connect(
        dbname="postgres",
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(DB_NAME)))
    print(f"✅ Datenbank '{DB_NAME}' wurde erstellt.")
    conn.close()

    # Neu verbinden mit der frisch erstellten DB
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cur = conn.cursor()
    print(f"🔁 Verbindung zu neuer Datenbank '{DB_NAME}' hergestellt.")


# === 3️⃣ Hilfsfunktion: prüfen, ob Tabelle existiert ===
def table_exists(table_name):
    cur.execute("""
        SELECT EXISTS (
            SELECT 1 FROM information_schema.tables
            WHERE table_name = %s
        );
    """, (table_name,))
    return cur.fetchone()[0]


# === 4️⃣ Tabelle: allegebaeude ===
if not table_exists("allegebaeude"):
    print("🧱 Erstelle Tabelle 'allegebaeude' ...")
    cur.execute("""
        CREATE TABLE allegebaeude (
            egid BIGINT PRIMARY KEY,
            gemeinde VARCHAR(100),
            kanton CHAR(2),
            baujahr SMALLINT,
            lat DOUBLE PRECISION NOT NULL,
            lon DOUBLE PRECISION NOT NULL,
            asbestrisiko SMALLINT NOT NULL DEFAULT 0,
            gebaeudestatus SMALLINT NOT NULL DEFAULT 0,
            gebaeudekategorie SMALLINT NOT NULL DEFAULT 0,
            erstellt_am TIMESTAMPTZ DEFAULT now(),
            erstellt_von VARCHAR(100),
            geaendert_am TIMESTAMPTZ DEFAULT now(),
            geaendert_von VARCHAR(100)
        );
        CREATE INDEX idx_allegebaeude_kanton       ON allegebaeude (kanton);
        CREATE INDEX idx_allegebaeude_gemeinde     ON allegebaeude (gemeinde);
        CREATE INDEX idx_allegebaeude_risiko       ON allegebaeude (asbestrisiko);
        CREATE INDEX idx_allegebaeude_status       ON allegebaeude (gebaeudestatus);
        CREATE INDEX idx_allegebaeude_kategorie    ON allegebaeude (gebaeudekategorie);
        CREATE INDEX idx_allegebaeude_coords       ON allegebaeude (lat, lon);
    """)
else:
    print("✅ Tabelle 'allegebaeude' existiert bereits.")


# === 5️⃣ Tabelle: kommentar_zu_gebaeuden ===
if not table_exists("kommentar_zu_gebaeuden"):
    print("🧱 Erstelle Tabelle 'kommentar_zu_gebaeuden")
    cur.execute("""
        CREATE TABLE kommentar_zu_gebaeuden (
            id BIGSERIAL PRIMARY KEY,
            egid BIGINT NOT NULL REFERENCES allegebaeude(egid) ON DELETE CASCADE,
            kommentar TEXT NOT NULL,
            neues_baujahr SMALLINT,
            neues_asbestrisiko SMALLINT,
            neues_lat DOUBLE PRECISION,
            neues_lon DOUBLE PRECISION,
            update_durch_user BOOLEAN DEFAULT FALSE,
            erstellt_am TIMESTAMPTZ DEFAULT now(),
            erstellt_von VARCHAR(100),
            geaendert_am TIMESTAMPTZ DEFAULT now(),
            geaendert_von VARCHAR(100)
        );
        CREATE INDEX idx_kommentar_egid_erstellt 
            ON kommentar_zu_gebaeuden (egid, erstellt_am DESC);
    """)
else:
    print("✅ Tabelle 'kommentar_zu_gebaeuden' existiert bereits.")


# === 6️⃣ Tabelle: updatereport ===
if not table_exists("updatereport"):
    print("🧱 Erstelle Tabelle 'updatereport' ...")
    cur.execute("""
        CREATE TABLE updatereport (
            id BIGSERIAL PRIMARY KEY,
            egid BIGINT NOT NULL,
            aktion VARCHAR(100) NOT NULL,
            beschreibung TEXT,
            status VARCHAR(50) DEFAULT 'nicht durchgeführt',
            erstellt_am TIMESTAMPTZ DEFAULT now(),
            erstellt_von VARCHAR(100)
        );
        CREATE INDEX idx_updatereport_egid ON updatereport (egid);
    """)
else:
    print("✅ Tabelle 'updatereport' existiert bereits.")

🔗 Verbinde mit PostgreSQL → admin@localhost:5433/suva2025
✅ Verbindung zu bestehender Datenbank 'suva2025' hergestellt.
🔗 Verbinde mit PostgreSQL → admin@localhost:5433/suva2025
✅ Verbindung zu bestehender Datenbank 'suva2025' hergestellt.
✅ Tabelle 'allegebaeude' existiert bereits.
✅ Tabelle 'kommentar_zu_gebaeuden' existiert bereits.
✅ Tabelle 'updatereport' existiert bereits.


# Datenquelle & Download

In [23]:
# === Fortschrittsbalken-Funktion mit ETA ===
def print_progress_bar(iteration, total, start_time, length=40):
    """
    Zeigt einen waagrechten Ladebalken mit ETA im Terminal.
    iteration : aktueller Fortschritt (z.B. Chunk-Nummer)
    total     : Gesamtzahl
    start_time: Zeitpunkt, an dem der Prozess begann (time.time())
    length    : Länge des Balkens in Zeichen
    """
    percent = iteration / total
    filled_length = int(length * percent)
    bar = "█" * filled_length + "░" * (length - filled_length)

    elapsed = time.time() - start_time
    if iteration > 0:
        estimated_total_time = elapsed / iteration * total
        remaining = estimated_total_time - elapsed
        eta = f"{int(remaining // 60):02d}:{int(remaining % 60):02d}"
    else:
        eta = "--:--"

    sys.stdout.write(f"\r[{bar}] {percent*100:5.1f}% ({iteration}/{total}) Geschä: {eta}")
    sys.stdout.flush()
    if iteration == total:
        print()  # Neue Zeile am Ende


# === Setup ===
datum = date.today().isoformat()

# Ordnerstruktur
os.makedirs("data/tmp_chunks", exist_ok=True)
final_csv_path = f"data/alle_relevanten_gebaeude_{datum}.csv"

base_url = "https://data.egid.ch/current.csv"
batch_size = 1000

# Grundquery (Filter <1990 oder kein Baujahr)
query = "SELECT * FROM building WHERE GBAUJ < 1990 OR GBAUJ = ''"

print("⬇️ Lade Gebäudedaten (<1990 oder ohne Baujahr) aus data.egid.ch ...\n")

session = requests.Session()

# === Schritt 1: Gesamtzahl bestimmen ===
test_sql = "SELECT count(*) FROM building WHERE GBAUJ < 1990 OR GBAUJ = ''"
encoded_test_sql = quote(test_sql)
test_url = f"{base_url}?format=csv&sql={encoded_test_sql}"

response = session.get(test_url)
response.raise_for_status()
total_count = int(pd.read_csv(StringIO(response.text)).iloc[0, 0])

print(f"📦 Gesamtanzahl Datensätze laut Server: {total_count:,}")
total_chunks = math.ceil(total_count / batch_size)
print(f"🧩 Erwartete Anzahl Chunks: {total_chunks:,}\n")

# === Schritt 2: Daten chunkweise laden ===
total_rows = 0
start_time = time.time()

for chunk_index in range(total_chunks):
    offset = chunk_index * batch_size
    sql = f"{query} ORDER BY EGID ASC LIMIT {batch_size} OFFSET {offset}"
    encoded_sql = quote(sql)
    url = f"{base_url}?format=csv&sql={encoded_sql}"

    response = session.get(url)
    response.raise_for_status()

    chunk = pd.read_csv(StringIO(response.text), sep=",", on_bad_lines="skip")

    if chunk.empty:
        print("\n⚠️ Leerer Chunk erhalten – möglicherweise Ende erreicht.")
        break

    total_rows += len(chunk)

    # Chunk speichern
    chunk_path = f"data/tmp_chunks/Chunk_{datum}_{chunk_index + 1:04d}.csv"
    chunk.to_csv(chunk_path, index=False)

    # Fortschrittsanzeige mit ETA
    print_progress_bar(chunk_index + 1, total_chunks, start_time, length=50)

print("\n📊 Alle Chunks heruntergeladen – bereit für Zusammenführung")

⬇️ Lade Gebäudedaten (<1990 oder ohne Baujahr) aus data.egid.ch ...

📦 Gesamtanzahl Datensätze laut Server: 2,553,898
🧩 Erwartete Anzahl Chunks: 2,554

[██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   4.4% (113/2554) Geschä: 05:01

KeyboardInterrupt: 

# Tagesfile erstellen & Chunk Temporär Dateien löschen

In [24]:
# Chunks zusammenführen ===
chunk_files = sorted(os.listdir("data/tmp_chunks"))
all_chunks = []

for f in chunk_files:
    path = os.path.join("data/tmp_chunks", f)
    df_part = pd.read_csv(path)
    all_chunks.append(df_part)

# Alles zusammenführen
df = pd.concat(all_chunks, ignore_index=True)

# 🔍 Duplikate zählen (nach EGID)
duplicate_count = df.duplicated(subset="EGID").sum()
print(f"\n🔁 Anzahl Duplikate (nach EGID): {duplicate_count:,}")

# Duplikate entfernen und sortieren
df = df.drop_duplicates(subset="EGID").sort_values("EGID")

# Speichern
df.to_csv(final_csv_path, index=False)

print(f"\n✅ Finale kombinierte Datei gespeichert: {final_csv_path}")
print(f"📦 Enthält {len(df):,} eindeutige Datensätze")

# === Schritt 4: Temporäre Chunk-Dateien löschen ===
for f in os.listdir("data/tmp_chunks"):
    os.remove(os.path.join("data/tmp_chunks", f))
os.rmdir("data/tmp_chunks")

print("\n🧹 Temporäre Chunk-Dateien gelöscht.")
print("🏁 Prozess abgeschlossen.")



🔁 Anzahl Duplikate (nach EGID): 0

✅ Finale kombinierte Datei gespeichert: data/alle_relevanten_gebaeude_2025-10-18.csv
📦 Enthält 113,000 eindeutige Datensätze

🧹 Temporäre Chunk-Dateien gelöscht.
🏁 Prozess abgeschlossen.


# Updatereport erzeugen (beim ersten Mal direkt CSV einlesen.ab hier noch weiter machen)

In [ ]:
## Ziel: Rückmeldungsübersicht
## Weitere überlegungen ab hier notwendig.... 
 
cur.execute("""
SELECT egid, gemeinde, kanton
FROM allegebaeude
WHERE geaendert_von = 'user';
""")

user_locked = cur.fetchall()

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(f"Update-Report für {today}\n")
    f.write("=" * 60 + "\n\n")
    f.write("Gebäude, die wegen Useränderungen NICHT überschrieben wurden:\n\n")
    if user_locked:
        for row in user_locked:
            f.write(f"EGID {row[0]} – {row[1]}, {row[2]}\n")
    else:
        f.write("Keine User-geschützten Datensätze gefunden.\n")

print(f"📄 Report erstellt: {REPORT_PATH}")



📄 Report erstellt: ./reports/updatereport_2025-10-18.txt


# PostGre SQL schliessen

In [ ]:
# === 7️⃣ Abschluss ===
conn.commit()
cur.close()
conn.close()

print("\n🏁 PostgreSQL-Setup vollständig abgeschlossen!")